In [1]:
import requests
import pandas as pd
import time
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import ast

API_KEY = "d801d36057b37f387632307ae6981fac"
BASE_URL = "https://api.themoviedb.org/3"

# Extract Movies from API

### Only need to run once

In [2]:
BASE_URL = "https://api.themoviedb.org/3"

# ----------------
# 1. Basic API request
# ----------------

def tmdb_get(endpoint, params=None, max_retries=3, timeout=20):
    if params is None:
        params = {}

    params = params.copy()
    params["api_key"] = API_KEY

    for attempt in range(max_retries):
        try:
            response = requests.get(
                BASE_URL + endpoint,
                params=params,
                timeout=timeout
            )

            if response.status_code == 429:
                wait_time = 2 ** attempt
                print(f"Rate limited. Waiting {wait_time} seconds...")
                time.sleep(wait_time)
                continue

            if response.status_code != 200:
                raise Exception(f"Error {response.status_code}: {response.text}")

            return response.json()

        except Exception as e:
            wait_time = 2 ** attempt

            if attempt == max_retries - 1:
                raise e

            print(f"Request failed. Retrying in {wait_time} seconds. Error: {e}")
            time.sleep(wait_time)

# ----------------
# 2. Check theatrical release
# ----------------

def has_us_regular_theatrical_release(details):
    """
    True if movie has regular US theatrical release.

    TMDb release types:
    1 = Premiere
    2 = Theatrical limited
    3 = Theatrical
    4 = Digital
    5 = Physical
    6 = TV
    """

    release_dates = details.get("release_dates", {}).get("results", [])

    for country in release_dates:
        if country.get("iso_3166_1") == "US":
            for release in country.get("release_dates", []):
                if release.get("type") == 3:
                    return True

    return False


def has_us_limited_or_regular_theatrical_release(details):
    """
    True if movie has limited or regular US theatrical release.
    """

    release_dates = details.get("release_dates", {}).get("results", [])

    for country in release_dates:
        if country.get("iso_3166_1") == "US":
            for release in country.get("release_dates", []):
                if release.get("type") in [2, 3]:
                    return True

    return False

# ----------------
# 3. Get movie details
# ----------------

def get_movie_details(movie_id):
    return tmdb_get(
        f"/movie/{movie_id}",
        params={
            "append_to_response": "credits,reviews,release_dates",
            "language": "en-US"
        }
    )

# ----------------
# 4. Parse details into one row
# ----------------

def parse_movie_details(details, max_cast=20):
    cast = details.get("credits", {}).get("cast", [])
    crew = details.get("credits", {}).get("crew", [])
    reviews = details.get("reviews", {}).get("results", [])

    directors = [
        person.get("name")
        for person in crew
        if person.get("job") == "Director" and person.get("name")
    ]

    top_cast = [
        person.get("name")
        for person in cast[:max_cast]
        if person.get("name")
    ]

    review_texts = [
        review.get("content", "")
        for review in reviews
        if review.get("content")
    ]

    production_companies = [
        company.get("name")
        for company in details.get("production_companies", [])
        if company.get("name")
    ]

    return {
        "tmdb_id": details.get("id"),
        "title": details.get("title"),
        "release_date": details.get("release_date"),
        "revenue": details.get("revenue"),
        "budget": details.get("budget"),
        "runtime": details.get("runtime"),
        "genres": [g["name"] for g in details.get("genres", []) if g.get("name")],
        "directors": directors,
        "top_cast": top_cast,
        "overview": details.get("overview"),
        "vote_average": details.get("vote_average"),
        "vote_count": details.get("vote_count"),
        "reviews": review_texts,
        "production_companies": production_companies,
        "us_regular_theatrical": has_us_regular_theatrical_release(details),
        "us_limited_or_regular_theatrical": has_us_limited_or_regular_theatrical_release(details)
    }

# ----------------
# 5. Get movie IDs from discover
# ----------------

def get_hollywood_movies_by_revenue(
    n_movies=10000,
    start_page=1,
    sleep_time=0,
    max_page=500,
    min_revenue=1_000_000,
    start_year=2000
):
    movies = []
    page = start_page

    while len(movies) < n_movies and page <= max_page:
        params = {
            "sort_by": "revenue.desc",
            "page": page,
            "include_adult": "false",
            "include_video": "false",
            "language": "en-US",
            "with_origin_country": "US",
            "with_original_language": "en",
            "region": "US",
            "primary_release_date.gte": f"{start_year}-01-01",
            "revenue.gte": min_revenue
        }

        data = tmdb_get("/discover/movie", params=params)
        results = data.get("results", [])

        if len(results) == 0:
            break

        movies.extend(results)

        page += 1
        time.sleep(sleep_time)

    return movies[:n_movies]


# ----------------
# 6. Fetch one movie in parallel
# ----------------

def fetch_one_movie_row(movie, max_cast=20, sleep_time=0, max_retries=3):
    movie_id = movie.get("id")

    for attempt in range(max_retries):
        try:
            details = get_movie_details(movie_id)
            row = parse_movie_details(details, max_cast=max_cast)

            time.sleep(sleep_time)

            return row

        except Exception as e:
            wait_time = 2 ** attempt

            if attempt == max_retries - 1:
                print(f"Failed permanently for movie ID {movie_id}: {e}")
                return None

            print(
                f"Attempt {attempt + 1} failed for movie ID {movie_id}. "
                f"Retrying in {wait_time}s. Error: {e}"
            )
            time.sleep(wait_time)


# ----------------
# 7. Build raw dataset
# ----------------

def extract_raw_movie_dataset(
    n_movies=10000,
    start_page=1,
    sleep_time=0,
    min_revenue=1_000_000,
    start_year=2000,
    max_cast=20,
    max_workers=50,
    output_csv="raw_tmdb_movies_2000_onward_min_1m.csv"
):
    movies = get_hollywood_movies_by_revenue(
        n_movies=n_movies,
        start_page=start_page,
        sleep_time=sleep_time,
        min_revenue=min_revenue,
        start_year=start_year
    )

    print(f"Movie IDs collected from discover: {len(movies)}")

    if len(movies) > 0:
        last_movie = movies[-1]
        print("\nLast movie collected from discover:")
        print("Rank:", len(movies))
        print("Title:", last_movie.get("title"))
        print("TMDb ID:", last_movie.get("id"))
        print("Release date:", last_movie.get("release_date"))
        print(f"Revenue: ${last_movie.get('revenue', 0):,}")

    print(f"\nFetching details using {max_workers} parallel workers...")

    rows = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(
                fetch_one_movie_row,
                movie,
                max_cast,
                sleep_time
            )
            for movie in movies
        ]

        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Fetching movie details"
        ):
            row = future.result()

            if row is not None:
                rows.append(row)

    raw_df = pd.DataFrame(rows)

    raw_df.to_csv(output_csv, index=False)

    print()
    print("Raw dataset shape:", raw_df.shape)
    print(f"Saved raw dataset to: {output_csv}")

    return raw_df

In [3]:
raw_movies_df = extract_raw_movie_dataset(
    n_movies=10000,
    start_page=1,
    sleep_time=0,
    min_revenue=1_000_000,
    start_year=2000,
    max_cast=20,
    max_workers=100,
    output_csv="raw_tmdb_movies_10k_2000_min_1m.csv"
)

Movie IDs collected from discover: 9998

Last movie collected from discover:
Rank: 9998
Title: If You Don't Have A Friend
TMDb ID: 1673094
Release date: 2020-08-12
Revenue: $0

Fetching details using 100 parallel workers...


Fetching movie details:   0%|          | 0/9998 [00:00<?, ?it/s]

Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waiting 1 seconds...
Rate limited. Waitin

# Filter raw dataset

### Can be rerun several times without having to do API call

In [4]:
def add_inflation_adjusted_money_columns(df, target_year=2026):
    df = df.copy()

    cpi_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL"
    cpi = pd.read_csv(cpi_url)

    cpi["observation_date"] = pd.to_datetime(cpi["observation_date"])
    cpi["year"] = cpi["observation_date"].dt.year
    cpi["CPIAUCSL"] = pd.to_numeric(cpi["CPIAUCSL"], errors="coerce")

    annual_cpi = (
        cpi.groupby("year", as_index=False)["CPIAUCSL"]
        .mean()
        .rename(columns={"CPIAUCSL": "cpi"})
    )

    # Use latest available 2026 CPI month
    target_cpi = cpi.loc[cpi["year"] == target_year, "CPIAUCSL"].dropna().iloc[-1]

    df["release_year"] = df["release_date"].dt.year

    df = df.merge(
        annual_cpi.rename(columns={"year": "release_year", "cpi": "release_year_cpi"}),
        on="release_year",
        how="left"
    )

    df["inflation_factor"] = target_cpi / df["release_year_cpi"]

    df["budget_adj_2026"] = df["budget"] * df["inflation_factor"]
    df["revenue_adj_2026"] = df["revenue"] * df["inflation_factor"]
    df["box_office_surplus_adj_2026"] = (
        df["revenue_adj_2026"] - df["budget_adj_2026"]
    )

    return df

In [8]:
import pandas as pd
import ast


# ----------------
# 1. Load raw dataset
# ----------------

raw_csv = "raw_tmdb_movies_10k_2000_min_1m.csv"
df = pd.read_csv(raw_csv)

print("Raw dataset shape:", df.shape)


# ----------------
# 2. Helpers
# ----------------

def parse_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    try:
        return ast.literal_eval(value)
    except:
        return []


def row_mentions_netflix(row):
    """
    Returns True if 'netflix' appears anywhere in the movie row.
    This checks all columns.
    """

    row_text_parts = []

    for value in row.values:
        if isinstance(value, list):
            row_text_parts.append(" ".join([str(x) for x in value]))
            continue

        if pd.isna(value):
            continue

        row_text_parts.append(str(value))

    full_text = " ".join(row_text_parts).lower()

    return "netflix" in full_text


# ----------------
# 3. Filter function
# ----------------

def filter_movie_dataset(
    df,
    start_year=2000,
    min_revenue=1_000_000,
    min_budget=1,
    use_regular_theatrical=True,
    use_limited_or_regular_theatrical=False,
    flag_netflix_mentions=True,
    manual_exclude_titles=None,
    output_csv="filtered_tmdb_movies.csv",
    netflix_raw_csv="movies_with_netflix_mentions_raw.csv",
    netflix_after_filtering_csv="movies_with_netflix_mentions_after_filtering.csv"
):
    if manual_exclude_titles is None:
        manual_exclude_titles = []

    if isinstance(manual_exclude_titles, str):
        manual_exclude_titles = [manual_exclude_titles]

    df = df.copy()

    # Parse list columns saved as strings
    for col in ["genres", "directors", "top_cast", "reviews", "production_companies"]:
        if col in df.columns:
            df[col] = df[col].apply(parse_list)

    netflix_output_columns = [
        "tmdb_id",
        "title",
        "release_date",
        "revenue",
        "budget"
    ]

    # ----------------
    # Netflix mention check BEFORE filtering
    # ----------------

    if flag_netflix_mentions:
        df["netflix_mentioned_anywhere_raw"] = df.apply(row_mentions_netflix, axis=1)

        netflix_raw_df = (
            df[df["netflix_mentioned_anywhere_raw"] == True]
            .copy()
        )

        netflix_raw_df["revenue"] = pd.to_numeric(netflix_raw_df["revenue"], errors="coerce")
        netflix_raw_df["budget"] = pd.to_numeric(netflix_raw_df["budget"], errors="coerce")

        netflix_raw_df = netflix_raw_df.sort_values(
            ["budget", "revenue"],
            ascending=[False, False]
        )

        netflix_raw_save_df = netflix_raw_df[
            [col for col in netflix_output_columns if col in netflix_raw_df.columns]
        ].copy()

        netflix_raw_save_df.to_csv(netflix_raw_csv, index=False)

        print(f"\nSaved raw movies with Netflix mentioned anywhere to: {netflix_raw_csv}")
        print("Movies with Netflix mentioned anywhere in raw data:", len(netflix_raw_save_df))

        print("\n=== Raw movies with Netflix mentioned anywhere ===")
        display(netflix_raw_save_df.head(30))

    else:
        df["netflix_mentioned_anywhere_raw"] = False
        netflix_raw_df = pd.DataFrame()

    # Convert columns
    df["revenue"] = pd.to_numeric(df["revenue"], errors="coerce")
    df["budget"] = pd.to_numeric(df["budget"], errors="coerce")
    df["runtime"] = pd.to_numeric(df["runtime"], errors="coerce")
    df["vote_average"] = pd.to_numeric(df["vote_average"], errors="coerce")
    df["vote_count"] = pd.to_numeric(df["vote_count"], errors="coerce")
    df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

    df = add_inflation_adjusted_money_columns(df, target_year=2026)

    df["us_regular_theatrical"] = (
        df["us_regular_theatrical"]
        .astype(str)
        .str.lower()
        .eq("true")
    )

    df["us_limited_or_regular_theatrical"] = (
        df["us_limited_or_regular_theatrical"]
        .astype(str)
        .str.lower()
        .eq("true")
    )

    rows_before = len(df)

    # Remove missing important values
    df = df.dropna(subset=[
        "tmdb_id",
        "title",
        "release_date",
        "revenue",
        "budget",
        "runtime",
        "genres",
        "directors",
        "top_cast",
        "overview",
        "vote_average",
        "vote_count",
        "reviews"
    ])

    rows_after_missing = len(df)

    # Remove duplicates
    df = df.drop_duplicates(subset=["tmdb_id"])
    rows_after_duplicates = len(df)

    # Remove future/unreleased
    today = pd.Timestamp.today()
    df = df[df["release_date"] <= today].copy()
    rows_after_release_date = len(df)

    # Year filter
    df = df[df["release_date"].dt.year >= start_year].copy()
    rows_after_start_year = len(df)

    # Numeric filters
    df = df[
        (df["revenue"] >= min_revenue) &
        (df["budget"] >= min_budget) &
        (df["runtime"] > 0) &
        (df["vote_count"] > 0)
    ].copy()

    rows_after_numeric = len(df)

    # Theatrical filters
    if use_regular_theatrical:
        df = df[df["us_regular_theatrical"] == True].copy()

    if use_limited_or_regular_theatrical:
        df = df[df["us_limited_or_regular_theatrical"] == True].copy()

    rows_after_theatrical = len(df)

    # Empty important list columns
    for col in ["genres", "directors", "top_cast"]:
        df = df[df[col].apply(lambda x: isinstance(x, list) and len(x) > 0)].copy()

    rows_after_empty_lists = len(df)

    # Calculate box-office surplus
    df["box_office_surplus"] = df["revenue"] - df["budget"]
    df["box_office_surplus_adj"] = df["revenue_adj_2026"] - df["budget_adj_2026"]

    # ----------------
    # Netflix mention check AFTER filtering
    # ----------------

    if flag_netflix_mentions:
        df["netflix_mentioned_anywhere_after_filtering"] = df.apply(
            row_mentions_netflix,
            axis=1
        )

        netflix_after_filtering_df = (
            df[df["netflix_mentioned_anywhere_after_filtering"] == True]
            .copy()
            .sort_values(["budget", "revenue"], ascending=[False, False])
        )

        netflix_after_filtering_save_df = netflix_after_filtering_df[
            [col for col in netflix_output_columns if col in netflix_after_filtering_df.columns]
        ].copy()

        netflix_after_filtering_save_df.to_csv(netflix_after_filtering_csv, index=False)

        print(f"\nSaved filtered movies with Netflix mentioned anywhere to: {netflix_after_filtering_csv}")
        print("Movies with Netflix mentioned anywhere after filtering:", len(netflix_after_filtering_save_df))

        print("\n=== Filtered movies with Netflix mentioned anywhere ===")
        display(netflix_after_filtering_save_df.head(30))

    else:
        df["netflix_mentioned_anywhere_after_filtering"] = False
        netflix_after_filtering_df = pd.DataFrame()

    # Manual exclusions
    if len(manual_exclude_titles) > 0:
        df = df[~df["title"].isin(manual_exclude_titles)].copy()

    rows_after_manual = len(df)

    # Sort
    #df = df.sort_values("box_office_surplus", ascending=False)
    df = df.sort_values("box_office_surplus_adj", ascending=False)

    # Save filtered dataset
    df.to_csv(output_csv, index=False)

    print("\nFiltering summary:")
    print("Rows before filtering:", rows_before)
    print("Rows after removing missing values:", rows_after_missing)
    print("Rows after removing duplicates:", rows_after_duplicates)
    print("Rows after removing future/unreleased movies:", rows_after_release_date)
    print(f"Rows after keeping movies from {start_year} onward:", rows_after_start_year)
    print("Rows after numeric filters:", rows_after_numeric)
    print("Rows after theatrical filter:", rows_after_theatrical)
    print("Rows after removing empty genres/directors/cast:", rows_after_empty_lists)

    if flag_netflix_mentions:
        print("Movies with Netflix mentioned anywhere in raw data:", len(netflix_raw_df))
        print("Movies with Netflix mentioned anywhere after filtering:", len(netflix_after_filtering_df))

    print("Rows after manual exclusions:", rows_after_manual)
    print()
    print("Final dataset shape:", df.shape)
    print(f"Saved filtered dataset to: {output_csv}")

    return df, netflix_raw_df, netflix_after_filtering_df


# ----------------
# 4. Run filtering
# ----------------

filtered_movies_df, netflix_raw_movies_df, netflix_after_filtering_movies_df = filter_movie_dataset(
    df,
    start_year=2000,
    min_revenue=1_000_000,
    min_budget=1,
    use_regular_theatrical=True,
    use_limited_or_regular_theatrical=False,
    flag_netflix_mentions=False,
    manual_exclude_titles=[
        "KPop Demon Hunters",
        "Glass Onion: A Knives Out Mystery",
        "Crouching Tiger, Hidden Dragon: Sword of Destiny"
    ],
    output_csv="top_10k_movies_2000_min_1m_adj_filtered.csv")

Raw dataset shape: (9997, 16)

Filtering summary:
Rows before filtering: 9997
Rows after removing missing values: 9949
Rows after removing duplicates: 9949
Rows after removing future/unreleased movies: 9819
Rows after keeping movies from 2000 onward: 9819
Rows after numeric filters: 4103
Rows after theatrical filter: 3784
Rows after removing empty genres/directors/cast: 3784
Rows after manual exclusions: 3781

Final dataset shape: (3781, 26)
Saved filtered dataset to: top_10k_movies_2000_min_1m_adj_filtered.csv
